In [ ]:
!pip install -U transformers accelerate bitsandbytes datasets peft qwen-vl-utils -q
print("Now click Runtime -> Restart Session, then run Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 51.5 MB/s eta 0:00:00
Now click Runtime -> Restart Session, then run Cell 2.


In [ ]:
# =====================================================================
# SIMPLE & RELIABLE FINE-TUNING v5 — Qwen2.5-VL-7B
# Saves directly to Drive + waits for sync + keeps 5 checkpoints
# =====================================================================

import os, glob, gc, time
import torch
from google.colab import drive
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from dataclasses import dataclass
from typing import Dict, List

# ======================== CONFIG ========================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
NUM_TRAIN_SAMPLES = 20000
MAX_SEQ_LEN = 768
MAX_PIXELS = 156800
SAVE_EVERY_N_STEPS = 25
MAX_STEPS = 2000
DRIVE_DIR = "/content/drive/MyDrive/GSV_Math_Model_Cache/qwen25vl_math_expert_finetuned"
ESSENTIAL_FILES = ["adapter_model.safetensors", "adapter_config.json", "trainer_state.json"]
# ========================================================

def is_valid_checkpoint(cp_path):
    for fname in ESSENTIAL_FILES:
        fpath = os.path.join(cp_path, fname)
        if not os.path.exists(fpath):
            return False
        if fname == "adapter_model.safetensors" and os.path.getsize(fpath) < 1_000_000:
            return False
    return True

# --- 0. VERIFY GPU IS CLEAN ---
gc.collect()
torch.cuda.empty_cache()
free_mem = torch.cuda.mem_get_info()[0] / 1e9
print(f"GPU Memory: {free_mem:.1f} GB free")
if free_mem < 12:
    raise RuntimeError("Not enough GPU memory! Click Runtime -> Restart Session first.")

# --- 1. MOUNT DRIVE ---
print("\n  Sign into the Google account that OWNS the GSV_Math_Model_Cache folder!\n")
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

# --- 2. LOAD MODEL ---
print("\nLoading Model in 4-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization_config, device_map={"": 0},
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
gc.collect()
torch.cuda.empty_cache()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
print(" Model ready.")

# --- 3. DATASET ---
print("\nLoading datasets...")
all_configs = get_dataset_config_names("lmms-lab/LLaVA-OneVision-Data")
math_configs = [c for c in all_configs if ('math' in c.lower() or 'geo' in c.lower() or 'science' in c.lower()) and 'clevr' not in c.lower()]

datasets_list = []
for config in math_configs[:3]:
    print(f"  {config}...")
    datasets_list.append(load_dataset("lmms-lab/LLaVA-OneVision-Data", config, split="train"))

dataset = concatenate_datasets(datasets_list).shuffle(seed=42).select(range(min(NUM_TRAIN_SAMPLES, len(concatenate_datasets(datasets_list)))))

def format_and_tokenize(batch):
    all_input_ids, all_attention_mask, all_labels = [], [], []
    for i in range(len(batch["conversations"])):
        try:
            img = batch["image"][i]
            convo = batch["conversations"][i]
            user_text = convo[0]["value"].replace("<image>", "").strip().replace("\\nHint:", "\nHint:")
            assistant_text = convo[1]["value"]
            messages_full = [{"role": "user", "content": [{"type": "image", "image": img, "max_pixels": MAX_PIXELS}, {"type": "text", "text": user_text}]}, {"role": "assistant", "content": [{"type": "text", "text": assistant_text}]}]
            messages_user = [{"role": "user", "content": [{"type": "image", "image": img, "max_pixels": MAX_PIXELS}, {"type": "text", "text": user_text}]}]
            full_text = processor.apply_chat_template(messages_full, tokenize=False, add_generation_prompt=False)
            user_text_only = processor.apply_chat_template(messages_user, tokenize=False, add_generation_prompt=True)
            full_enc = processor(text=[full_text], images=[img], padding=False, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
            user_enc = processor(text=[user_text_only], images=[img], padding=False, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
            input_ids, attention_mask = full_enc["input_ids"][0], full_enc["attention_mask"][0]
            labels = input_ids.clone()
            labels[:user_enc["input_ids"].shape[1]] = -100
            all_input_ids.append(input_ids)
            all_attention_mask.append(attention_mask)
            all_labels.append(labels)
        except Exception:
            continue
    return {"input_ids": all_input_ids, "attention_mask": all_attention_mask, "labels": all_labels}

print("Tokenizing...")
train_dataset = dataset.map(format_and_tokenize, batched=True, batch_size=10, num_proc=1, remove_columns=dataset.column_names)
print(f" {len(train_dataset)} examples ready.")

# --- 4. DATA COLLATOR ---
@dataclass
class VLMDataCollator:
    pad_token_id: int = 0
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids_padded, attention_mask_padded, labels_padded = [], [], []
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids_padded.append(torch.cat([torch.tensor(f["input_ids"]), torch.full((pad_len,), self.pad_token_id)]))
            attention_mask_padded.append(torch.cat([torch.tensor(f["attention_mask"]), torch.zeros(pad_len, dtype=torch.long)]))
            labels_padded.append(torch.cat([torch.tensor(f["labels"]), torch.full((pad_len,), -100)]))
        return {"input_ids": torch.stack(input_ids_padded), "attention_mask": torch.stack(attention_mask_padded), "labels": torch.stack(labels_padded)}

# --- 5. WAIT-FOR-SYNC CALLBACK ---
class WaitForSyncCallback(TrainerCallback):
    """After each save, wait 30 seconds for Google Drive FUSE to sync the files."""
    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        cp_path = os.path.join(DRIVE_DIR, f"checkpoint-{step}")
        print(f"\n   Saved checkpoint-{step}. Waiting 30s for Drive sync...", end="", flush=True)
        time.sleep(30)  # Give FUSE time to push to Google servers
        if is_valid_checkpoint(cp_path):
            size = os.path.getsize(os.path.join(cp_path, "adapter_model.safetensors")) / 1e6
            print(f"  SYNCED! ({size:.1f} MB)")
        else:
            print(f"  May not be fully synced yet. Older checkpoints are still safe.")

# --- 6. FIND VALID CHECKPOINT TO RESUME FROM ---
print("\nScanning Drive for valid checkpoints...")
all_cps = sorted(glob.glob(f"{DRIVE_DIR}/checkpoint-*"))

for cp in all_cps:
    valid = is_valid_checkpoint(cp)
    if valid:
        size = os.path.getsize(os.path.join(cp, "adapter_model.safetensors")) / 1e6
        print(f"   {os.path.basename(cp)} ({size:.1f} MB)")
    else:
        print(f"   {os.path.basename(cp)} CORRUPTED — deleting...")
        import shutil
        shutil.rmtree(cp)

valid_cps = sorted([cp for cp in glob.glob(f"{DRIVE_DIR}/checkpoint-*") if is_valid_checkpoint(cp)])
resume_from = valid_cps[-1] if valid_cps else None

if resume_from:
    print(f"\n   Will resume from: {os.path.basename(resume_from)}")
else:
    print(f"\n   No valid checkpoints. Starting fresh.")

# --- 7. TRAINER ---
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    data_collator=VLMDataCollator(pad_token_id=processor.tokenizer.pad_token_id or 0),
    callbacks=[WaitForSyncCallback()],
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="paged_adamw_8bit",
        output_dir=DRIVE_DIR,          # Save DIRECTLY to Drive
        save_strategy="steps",
        save_steps=SAVE_EVERY_N_STEPS,
        save_total_limit=5,            # Keep 5 checkpoints for safety
        report_to="none",
        gradient_checkpointing=True,
        remove_unused_columns=False,
    ),
)

# --- 8. TRAIN ---
if resume_from:
    print(f"\n Resuming from {os.path.basename(resume_from)}...")
    trainer.train(resume_from_checkpoint=resume_from)
else:
    print("\n Starting training!")
    trainer.train()

# --- 9. SAVE FINAL ---
model.save_pretrained(DRIVE_DIR)
processor.save_pretrained(DRIVE_DIR)
time.sleep(30)
print(f"\n Done! Model saved to {DRIVE_DIR}")

GPU Memory: 15.5 GB free

  Sign into the Google account that OWNS the GSV_Math_Model_Cache folder!

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loading Model in 4-bit...


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Map (num_proc=1):   0%|          | 0/20000 [00:00<?, ? examples/s]

 20000 examples ready.

Scanning Drive for valid checkpoints...
   checkpoint-1875 (190.4 MB)
   checkpoint-1900 (190.4 MB)
   checkpoint-1925 (190.4 MB)
   checkpoint-1950 (190.4 MB)
   checkpoint-1975 (190.4 MB)

   Will resume from: checkpoint-1975

 Resuming from checkpoint-1975...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1980,0.380347
1985,0.379744
1990,0.311803
1995,0.259372
2000,0.286379



   Saved checkpoint-2000. Waiting 30s for Drive sync...  SYNCED! (190.4 MB)

 Done! Model saved to /content/drive/MyDrive/GSV_Math_Model_Cache/qwen25vl_math_expert_finetuned


In [ ]:
import os, json, time, re
from tqdm.auto import tqdm
from PIL import Image
from qwen_vl_utils import process_vision_info
from datasets import load_dataset
import torch

# =====================================================================
# 1. DEFINE THE SMART PARSER (So the notebook knows how to grade)
# =====================================================================
FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches: return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    for prefix in ["the answer is", "therefore, the answer is", "so the answer is", "the value is", "answer is", "value is", "equals", "it is", "the final answer is", "final answer:", "answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    if not choices: return extraction
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
        else:
            extraction = cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if str(pred).lower().strip() == str(gt).lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

# =====================================================================
# 2. RUN THE VDS BLIND ABLATION
# =====================================================================
print("Loading MathVista testmini for VDS Ablation...")
eval_dataset = load_dataset("AI4Math/MathVista", split="testmini")

print("Starting Blind Ablation (VDS) pass for Qwen...")
blank_image = Image.new('RGB', (224, 224), color='black')

VDS_RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen_zeroshot_BLIND.json"
vds_results = []
if os.path.exists(VDS_RESULTS_FILE):
    with open(VDS_RESULTS_FILE, "r") as f:
        vds_results = json.load(f)
completed_pids = {res["pid"] for res in vds_results}

remaining_samples = [s for s in eval_dataset if s["pid"] not in completed_pids]

for sample in tqdm(remaining_samples):
    pid = sample["pid"]
    question = sample["query"]
    gt = sample["answer"]

    # Pass the BLANK image
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": blank_image},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    parsed_ans = normalize_extracted_answer(output_text, sample.get("choices", []), sample["question_type"], sample["answer_type"])
    correct = is_correct(parsed_ans, gt, sample["answer_type"])

    vds_results.append({
        "pid": pid, "question": question, "raw_response": output_text,
        "parsed_response": parsed_ans, "ground_truth": gt, "correct": correct
    })

    if len(vds_results) % 10 == 0:
        with open(VDS_RESULTS_FILE, "w") as f:
            json.dump(vds_results, f, indent=4)

with open(VDS_RESULTS_FILE, "w") as f:
    json.dump(vds_results, f, indent=4)

print(f"BLIND ACCURACY: {(sum(r['correct'] for r in vds_results) / len(vds_results)) * 100:.2f}%")

Loading MathVista testmini for VDS Ablation...
Starting Blind Ablation (VDS) pass for Qwen...


  0%|          | 0/1000 [00:00<?, ?it/s]

BLIND ACCURACY: 16.30%


In [1]:
# =====================================================================
# ONE-CLICK HOLDOUT EVALUATION (LOADS FROM DRIVE & TESTS)
# =====================================================================
# 1. Install dependencies for the fresh Colab runtime
!pip install unsloth unsloth_zoo trl peft bitsandbytes accelerate
!pip install --upgrade transformers datasets

import os, json, gc
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from PIL import Image
from unsloth import FastVisionModel

# --- CONFIGURATION ---
# TODO: Put the exact path to your saved LoRA checkpoint on Google Drive here!
CHECKPOINT_PATH = "/content/drive/MyDrive/GSV_Math_Model_Cache/qwen25vl_math_expert_finetuned/checkpoint-2000"
HOLDOUT_RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/holdout_results.json"
NUM_SAMPLES = 500

print(f"\n1. Loading Finetuned Model directly from: {CHECKPOINT_PATH}")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = CHECKPOINT_PATH,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth"
)
FastVisionModel.for_inference(model)

print("\n2. Loading MathV360K (GeoQA subset) holdout...")
train_dataset = load_dataset("lmms-lab/LLaVA-OneVision-Data", "GeoQA+(MathV360K)", split="train")

# Slice the LAST 500 samples to guarantee it wasn't used in your training
holdout_set = train_dataset.select(range(len(train_dataset) - NUM_SAMPLES, len(train_dataset)))
print(f"Evaluating on {len(holdout_set)} unseen training samples...")

os.makedirs(os.path.dirname(HOLDOUT_RESULTS_FILE), exist_ok=True)
holdout_results = []

print("\n3. Starting Evaluation Loop...")
for idx, sample in enumerate(tqdm(holdout_set)):
    # Parse dataset format
    conversations = sample["conversations"]
    user_msg = next(msg["value"] for msg in conversations if msg["from"] == "human")
    question = user_msg.replace("<image>\n", "").replace("<image>", "").strip()
    ground_truth = next(msg["value"] for msg in conversations if msg["from"] == "gpt").strip()

    # VRAM Protection: Resize image
    image = sample["image"]
    if max(image.size) > 768:
        ratio = 768 / max(image.size)
        image = image.resize((int(image.size[0] * ratio), int(image.size[1] * ratio)), Image.Resampling.LANCZOS)

    # Inference
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], images=[image], return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
        response = tokenizer.decode(out_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

    # Grading (Strict exact match for validation)
    is_correct = 1 if ground_truth.lower() in response.lower() else 0

    holdout_results.append({
        "id": f"holdout_{idx}",
        "question": question,
        "raw_response": response,
        "ground_truth": ground_truth,
        "correct": is_correct
    })

    # VRAM Protection: Memory cleanup
    del inputs, out_ids
    torch.cuda.empty_cache()
    gc.collect()

    # Save safely
    if len(holdout_results) % 10 == 0:
        with open(HOLDOUT_RESULTS_FILE, "w") as f:
            json.dump(holdout_results, f, indent=4)

# Final save
with open(HOLDOUT_RESULTS_FILE, "w") as f:
    json.dump(holdout_results, f, indent=4)

accuracy = sum(r["correct"] for r in holdout_results) / len(holdout_results) * 100
print("="*50)
print(f"HOLDOUT ACCURACY (UNSEEN TRAIN DATA): {accuracy:.2f}%")
print("="*50)

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.1
    Uninstalling datasets-5.0.1:
      Successfully uninstalled datasets-5.0.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1
  Using cached transformers-5.15.1-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
Using cached transformers-5.15.1-py3-none-any.whl (11.7 MB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
ERROR: pi

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


2. Loading MathV360K (GeoQA subset) holdout...
Evaluating on 500 unseen training samples...

3. Starting Evaluation Loop...


  0%|          | 0/500 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

HOLDOUT ACCURACY (UNSEEN TRAIN DATA): 94.00%


In [2]:
import json
from collections import defaultdict

HOLDOUT_RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/holdout_results.json"

with open(HOLDOUT_RESULTS_FILE, "r") as f:
    results = json.load(f)

def categorize(question):
    q = question.lower()
    if "area" in q or "volume" in q or "perimeter" in q: return "Area / Volume"
    if "angle" in q or "degree" in q: return "Angles"
    if "length" in q or "distance" in q or "find x" in q: return "Lengths & Algebra"
    if "ratio" in q or "percentage" in q or "probability" in q: return "Stats & Ratios"
    return "General Geometry"

cat_stats = defaultdict(lambda: {"total": 0, "correct": 0})
for r in results:
    cat = categorize(r["question"])
    cat_stats[cat]["total"] += 1
    cat_stats[cat]["correct"] += r["correct"]

print("="*60)
print(f"{'Category':<25} | {'Total':<6} | {'Accuracy'}")
print("-" * 60)
for cat, stats in sorted(cat_stats.items()):
    acc = (stats["correct"] / stats["total"]) * 100
    print(f"{cat:<25} | {stats['total']:<6} | {acc:.2f}%")
print("="*60)

Category                  | Total  | Accuracy
------------------------------------------------------------
Angles                    | 12     | 100.00%
Area / Volume             | 6      | 83.33%
General Geometry          | 479    | 93.95%
Lengths & Algebra         | 3      | 100.00%


In [3]:
# =====================================================================
# VDS: BLIND ACCURACY (MATHVISTA) - NO IMAGES PROVIDED
# =====================================================================
import json, os, gc, re
from tqdm.auto import tqdm
from datasets import load_dataset
import torch

dataset = load_dataset("AI4Math/MathVista", split="testmini")

BLIND_RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/vds_blind_results.json"
os.makedirs(os.path.dirname(BLIND_RESULTS_FILE), exist_ok=True)

def extract_final_answer(text):
    match = re.search(r'(?:[Tt]he answer is|answer:|choose)\s*([A-D0-9\.\-\/]+)', text)
    if match: return match.group(1).strip()
    nums = re.findall(r'-?\d*\.?\d+', text)
    return nums[-1] if nums else text[-10:].strip()

blind_results = []
if os.path.exists(BLIND_RESULTS_FILE):
    with open(BLIND_RESULTS_FILE, "r") as f:
        blind_results = json.load(f)

completed = {r["pid"] for r in blind_results}
remaining = [s for s in dataset if s["pid"] not in completed]

print(f"Resuming: {len(completed)}/1000 blind evaluations complete.")

for sample in tqdm(remaining):
    question = sample["query"]
    ground_truth = sample["answer"]

    # CRITICAL FOR VDS: Text-only prompt, no image!
    messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
        response = tokenizer.decode(out_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

    ans = extract_final_answer(response)

    blind_results.append({
        "pid": sample["pid"],
        "question": question,
        "blind_response": response,
        "extracted_answer": ans,
        "ground_truth": ground_truth,
    })

    # VRAM Cleanup
    del inputs, out_ids
    torch.cuda.empty_cache()
    gc.collect()

    if len(blind_results) % 20 == 0:
        with open(BLIND_RESULTS_FILE, "w") as f:
            json.dump(blind_results, f, indent=4)

with open(BLIND_RESULTS_FILE, "w") as f:
    json.dump(blind_results, f, indent=4)

print(" Blind Evaluation Complete! Ready to calculate VDS.")

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…): reconstructing file:   0%|          |  0.00B /  142MB            

data/testmini-00000-of-00001-725687bf7a1(…): downloading bytes:           |  0.00B            

data/test-00000-of-00002-6b81bd7f7e2065e(…): reconstructing file:   0%|          |  0.00B /  358MB            

data/test-00000-of-00002-6b81bd7f7e2065e(…): downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

Resuming: 0/1000 blind evaluations complete.


  0%|          | 0/1000 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

 Blind Evaluation Complete! Ready to calculate VDS.
